# ParametrizANI — RESP Charges with Psi4

**Perform geometry optimization with TorchANI/AIMNet2/GFN2-xTB, then compute RESP charges with Psi4 to generate GAFF2 topology parameters.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pablo-arantes/ParametrizANI/blob/main/ParametrizANI_RESP_charges.ipynb)

---

## Workflow Overview

1. **Install dependencies** — ParametrizANI + Psi4 + resp
2. **Input molecule** — SMILES, PDB, or MOL file
3. **Geometry optimization** — Optimize with ML potential (TorchANI, AIMNet2, etc.)
4. **RESP charge calculation** — Compute QM-quality charges with Psi4
5. **GAFF2 topology generation** — Build topology using Psi4 RESP charges
6. **Export** — AMBER, GROMACS, or OpenMM formats

---

### Reference

> Arantes et al. "ParametrizANI: Fast and Accessible Dihedral Parametrization for Small Molecules."
> *J. Chem. Inf. Model.* (2025) doi: [10.1021/acs.jcim.5c01957](http://pubs.acs.org/doi/abs/10.1021/acs.jcim.5c01957)

In [ ]:
#@title **1. Install Dependencies**
#@markdown This cell installs ParametrizANI and its dependencies.
#@markdown Psi4 is installed in a separate conda environment.

import os, sys

# Install ParametrizANI package
!pip install -q torchani ase rdkit-pypi parmed
!pip install -e /content/ParametrizANI_package 2>/dev/null || pip install parametrizani

# Install AmberTools (for antechamber/tLEaP)
!conda install -y -c conda-forge ambertools &> /dev/null

# Install Psi4 + resp in a separate environment (Colab-compatible)
!conda install -y -c conda-forge psi4 resp &> /dev/null

print("\u2713 All dependencies installed")

In [ ]:
#@title **2. Set Working Directory**

workDir = '/content/resp_workflow' #@param {type:"string"}
os.makedirs(workDir, exist_ok=True)
print(f"Working directory: {workDir}")

In [ ]:
#@title **3. Imports**

import numpy as np
import matplotlib.pyplot as plt
from parametrizani import (
    ConformerGenerator,
    TopologyGenerator,
    calculate_resp_charges,
    get_dihedral_atom_types,
)
print("\u2713 Imports successful")

In [ ]:
#@title **4. Molecule Input**
#@markdown Enter your molecule as SMILES, or provide a file path (PDB/MOL).
Type = "smiles" #@param ["smiles", "pdb", "mol"]
smiles_or_filename = "COc3ccc2c(=O)cc(c1ccccc1)oc2c3" #@param {type:"string"}

gen = ConformerGenerator(smiles_or_filename, Type, workDir)
conf = gen.run()
print(f"\u2713 Molecule: {conf['smiles']} ({conf['num_atoms']} atoms)")
print(f"  MOL file: {conf['mol_file']}")
print(f"  PDB file: {conf['pdb_file']}")

# 3D visualization (optional)
try:
    import py3Dmol
    with open(conf['pdb_file']) as f:
        pdb_data = f.read()
    view = py3Dmol.view(width=400, height=300)
    view.addModel(pdb_data, "pdb")
    view.setStyle({'stick': {}})
    view.addPropertyLabels("index", {}, {'fontColor': 'black', 'fontSize': 10})
    view.zoomTo()
    view.show()
except ImportError:
    print("  (Install py3Dmol for 3D visualization)")

---

## Step 1: Geometry Optimization

Optimize the molecular geometry using an ML potential before computing RESP charges.
This ensures we compute charges on a realistic, low-energy structure.

Available methods: `torchani` (ANI-2x), `ani2xr`, `ani2dr`, `aimnet2`, `mace`, `xtb`

In [ ]:
#@title **5. Geometry Optimization**
#@markdown Select ML potential for geometry optimization:
opt_method = "torchani" #@param ["torchani", "ani2x", "ani1x", "ani1ccx", "ani2xr", "ani2dr", "ani2xr_snn", "ani_mbis", "aimnet2", "wb97m_d3", "b97_3c", "mace", "xtb"]
#@markdown Convergence threshold (fmax in eV/\u00c5):
fmax = 0.001 #@param {type:"number"}

from parametrizani import ReferenceEnergyCalculator
from ase.io import read as ase_read
from ase.optimize import BFGS
import torch

# Read molecule with ASE
atoms = ase_read(conf['mol_file'])

# Set up ML calculator
calc = ReferenceEnergyCalculator(opt_method, workDir)
atoms.calc = calc._get_calculator()

# Optimize geometry
opt = BFGS(atoms, logfile=os.path.join(workDir, 'geom_opt.log'))
opt.run(fmax=fmax)

# Save optimized structure
from ase.io import write as ase_write
opt_mol_file = os.path.join(workDir, 'optimized_molecule.mol')
opt_pdb_file = os.path.join(workDir, 'optimized_molecule.pdb')
ase_write(opt_mol_file, atoms, format='mol')
ase_write(opt_pdb_file, atoms, format='proteindatabank')

print(f"\u2713 Geometry optimized with {opt_method} (fmax={fmax})")
print(f"  Optimized MOL: {opt_mol_file}")
print(f"  Final energy: {atoms.get_potential_energy():.6f} eV")
print(f"  Optimization steps: {opt.nsteps}")

---

## Step 2: RESP Charge Calculation with Psi4

Compute Restrained Electrostatic Potential (RESP) charges using Psi4.

- **Standard protocol:** HF/6-31G* (Bayly et al. 1993, matches AMBER convention)
- **Higher accuracy:** B3LYP/cc-pVTZ or MP2/cc-pVDZ

In [ ]:
#@title **6. Calculate RESP Charges**
#@markdown QM method for ESP calculation:
resp_method = "HF" #@param ["HF", "B3LYP", "MP2", "wB97X-D", "PBE0", "M06-2X"]
#@markdown Basis set:
resp_basis = "6-31G*" #@param ["6-31G*", "6-31G**", "cc-pVDZ", "cc-pVTZ", "aug-cc-pVDZ", "def2-SVP", "def2-TZVP"]
#@markdown Net molecular charge:
mol_charge = 0 #@param {type:"integer"}
#@markdown Spin multiplicity:
mol_multiplicity = 1 #@param {type:"integer"}
#@markdown Psi4 memory:
psi4_memory = "4 GB" #@param ["2 GB", "4 GB", "8 GB", "16 GB"]
#@markdown Number of CPU threads:
psi4_threads = 2 #@param [1, 2, 4, 8] {type:"raw"}

# Use the optimized geometry for RESP calculation
resp_input_file = opt_mol_file if os.path.exists(opt_mol_file) else conf['mol_file']

resp_result = calculate_resp_charges(
    resp_input_file,
    method=resp_method,
    basis=resp_basis,
    charge=mol_charge,
    multiplicity=mol_multiplicity,
    memory=psi4_memory,
    n_threads=int(psi4_threads),
)

print(f"\u2713 RESP charges calculated ({resp_method}/{resp_basis})")
print(f"  Input file: {resp_input_file}")
print(f"  Output file: {resp_result['output_file']}")
print(f"\n  {'Atom':<6} {'Charge':>10}")
print(f"  {'-'*6} {'-'*10}")
for i, (symbol, q) in enumerate(zip(resp_result['atom_symbols'], resp_result['charges'])):
    print(f"  {i:3d} {symbol:2s}  {q:10.6f}")
print(f"\n  Total charge: {resp_result['charges'].sum():.6f}")
print(f"  Number of atoms: {resp_result['num_atoms']}")

---

## Step 3: GAFF2 Topology Generation

Generate AMBER topology files using the Psi4 RESP charges.
The `resp_charges_file` parameter tells antechamber to read the pre-computed charges
directly (`-c rc -cf <file>`) instead of computing them internally.

In [ ]:
#@title **7. Generate GAFF2 Topology with RESP Charges**
#@markdown Output format:
output_format = "AMBER" #@param ["AMBER", "GROMACS", "OpenMM", "All"]

topo = TopologyGenerator(workDir, force_field='gaff2')

# Generate AMBER topology using Psi4 RESP charges
amber_files = topo.generate_amber(
    resp_input_file,
    resp_charges_file=resp_result['output_file']  # Uses Psi4 charges via antechamber -c rc
)

print(f"\u2713 GAFF2 topology generated with Psi4 RESP charges")
print(f"  PRMTOP: {amber_files['prmtop']}")
print(f"  INPCRD: {amber_files['inpcrd']}")
print(f"  MOL2:   {amber_files['mol2']}")
print(f"  FRCMOD: {amber_files['frcmod']}")

# Export to other formats if requested
if output_format in ["GROMACS", "All"]:
    try:
        gmx = topo.generate_gromacs(amber_files['prmtop'], amber_files['inpcrd'])
        print(f"\n\u2713 GROMACS files:")
        print(f"  TOP: {gmx['top']}")
        print(f"  GRO: {gmx['gro']}")
    except Exception as e:
        print(f"  GROMACS export: {e}")

if output_format in ["OpenMM", "All"]:
    try:
        omm = topo.generate_openmm(amber_files['prmtop'], amber_files['inpcrd'])
        print(f"\n\u2713 OpenMM files:")
        print(f"  XML: {omm['xml']}")
        print(f"  PDB: {omm['pdb']}")
    except Exception as e:
        print(f"  OpenMM export: {e}")

---

## Step 4: Verify Charges in Topology

Confirm that the RESP charges were correctly incorporated into the topology.

In [ ]:
#@title **8. Verify Charges in MOL2**
#@markdown Compare Psi4 RESP charges with those embedded in the topology MOL2 file.

# Read charges from the generated MOL2
mol2_charges = []
mol2_atoms = []
with open(amber_files['mol2'], 'r') as f:
    in_atom_block = False
    for line in f:
        if line.startswith('@<TRIPOS>ATOM'):
            in_atom_block = True
            continue
        if line.startswith('@<TRIPOS>'):
            in_atom_block = False
            continue
        if in_atom_block:
            parts = line.split()
            if len(parts) >= 9:
                mol2_atoms.append(parts[1])
                mol2_charges.append(float(parts[8]))

mol2_charges = np.array(mol2_charges)

# Compare
print(f"{'Atom':<8} {'Psi4 RESP':>12} {'MOL2':>12} {'Diff':>10}")
print(f"{'-'*8} {'-'*12} {'-'*12} {'-'*10}")
for i, (sym, q_psi4, q_mol2) in enumerate(zip(resp_result['atom_symbols'], resp_result['charges'], mol2_charges)):
    diff = abs(q_psi4 - q_mol2)
    flag = ' ***' if diff > 0.001 else ''
    print(f"{i:3d} {sym:2s}   {q_psi4:12.6f} {q_mol2:12.6f} {diff:10.6f}{flag}")

max_diff = np.max(np.abs(resp_result['charges'] - mol2_charges))
print(f"\nMax charge difference: {max_diff:.6f}")
if max_diff < 0.01:
    print("\u2705 Charges correctly transferred to topology!")
else:
    print("\u26a0\ufe0f Significant charge differences detected — check antechamber output.")

---

## (Optional) Dihedral Parametrization

If you also want to optimize dihedral parameters using the RESP-charged topology,
continue with the standard ParametrizANI workflow below.

In [ ]:
#@title **9. (Optional) Set Dihedral for Parametrization**
#@markdown Atom indices (0-based) defining the dihedral to optimize:
atom1 = 8 #@param {type:"integer"}
atom2 = 9 #@param {type:"integer"}
atom3 = 10 #@param {type:"integer"}
atom4 = 15 #@param {type:"integer"}
#@markdown Scan step size (degrees):
degrees_steps = 30 #@param [5, 10, 15, 20, 30, 45, 60] {type:"raw"}

dihedral_indices = [atom1, atom2, atom3, atom4]

# Generate dihedral conformers
scan = gen.generate_dihedral_conformers(dihedral_indices, step=int(degrees_steps))
print(f"\u2713 Generated {len(scan['angles'])} conformers")
print(f"  Dihedral: atoms {dihedral_indices}")

# Detect atom types from MOL2
atom_types = get_dihedral_atom_types(amber_files['mol2'], dihedral_indices)
print(f"  Atom types: {atom_types}")

In [ ]:
#@title **10. (Optional) Reference Energy & Optimization**
#@markdown ML method for reference energies:
ref_method = "torchani" #@param ["torchani", "ani2x", "ani2xr", "ani2dr", "aimnet2", "mace", "xtb"]
#@markdown Maximum Fourier terms:
max_terms = 4 #@param [1, 2, 3, 4, 5, 6] {type:"raw"}
#@markdown Convergence threshold:
opt_fmax = 0.001 #@param {type:"number"}

from parametrizani import (
    ReferenceEnergyCalculator,
    EnergyMinimizer,
    DihedralOptimizer,
)

# Reference energies
ref_calc = ReferenceEnergyCalculator(ref_method, workDir)
ref = ref_calc.scan_dihedral(
    scan['conformers'], scan['angles'],
    optimize=True, fmax=opt_fmax,
    dihedral_indices=dihedral_indices
)
print(f"\u2713 Reference energy range: {max(ref['energies_relative']):.3f} kcal/mol")

# MM minimization with dihedral zeroed
minimizer = EnergyMinimizer('gaff2', workDir)
mm = minimizer.minimize_scan(
    amber_files['prmtop'], amber_files['inpcrd'],
    scan['pdb_files'], dihedral_indices,
    angles=scan['angles'], zero_dihedral=True
)
print(f"\u2713 MM energy range (dihedral zeroed): {max(mm['energies_relative']):.3f} kcal/mol")

# Optimize dihedral parameters
optimizer = DihedralOptimizer(max_terms=int(max_terms), work_dir=workDir)
result = optimizer.run_optimization(
    ref['angles'], ref['energies_relative'],
    mm_energies=mm['energies_relative'],
    atom_types=atom_types
)

print(f"\n\u2713 Optimization complete!")
print(f"  Best RMSE ({max_terms} terms): {result['rmse']:.4f} kcal/mol")
print(f"\n  RMSE per number of terms:")
for i, rmse in enumerate(result['all_rmse'], 1):
    print(f"    {i} terms: {rmse:.4f} kcal/mol")
print(f"\n  Optimized FRCMOD Parameters:")
print(f"  {result['frcmod_params']}")

# Plot
min_deg, max_deg = min(ref['angles']), max(ref['angles'])
plt.figure(figsize=(8, 5))
plt.plot(ref['angles'], ref['energies_relative'], 'o-', linewidth=1.5, label=ref_method)
plt.plot(mm['angles'], mm['energies_relative'], 's--', linewidth=1.0, label="GAFF2 (RESP, original)", alpha=0.7)
plt.plot(result['angles'], result['best_fit'], 'D-', linewidth=1.5, label="Optimized")
plt.xticks(np.arange(min_deg, max_deg + 1, 60.0))
plt.xlabel('Dihedral Angle (degrees)')
plt.ylabel('Relative Energy (kcal/mol)')
plt.legend(frameon=True)
plt.title(f'Dihedral Optimization with RESP Charges (RMSE: {result["rmse"]:.4f} kcal/mol)')
plt.tight_layout()
plt.savefig(os.path.join(workDir, 'optimized_resp_profile.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#@title **11. (Optional) Write FRCMOD & Generate Final Topology**
#@markdown IDIVF (scaling factor for equivalent torsions):
IDIVF = 1 #@param [1, 2, 3, 4] {type:"raw"}
#@markdown Number of Fourier terms:
n_terms = 4 #@param [1, 2, 3, 4, 5, 6] {type:"raw"}
#@markdown Output format:
final_format = "AMBER" #@param ["AMBER", "GROMACS", "OpenMM"]

selected_frcmod_params = optimizer.format_frcmod_params(
    result, n_terms=int(n_terms), idivf=int(IDIVF)
)
frcmod_file = optimizer.write_frcmod(result, idivf=int(IDIVF), n_terms=int(n_terms))
print(f"\u2713 FRCMOD written: {frcmod_file}")

# Update FRCMOD with optimized parameters
topology_mol2 = os.path.join(topo.tleap_dir, 'new.mol2')
if not os.path.exists(topology_mol2):
    topology_mol2 = amber_files['mol2']

updated_frcmod = topo.update_frcmod(
    amber_files['frcmod'],
    selected_frcmod_params,
    dihedral_indices=dihedral_indices,
    mol2_file=topology_mol2,
)
print(f"\u2713 Updated FRCMOD: {updated_frcmod}")

# Generate final topology
new_amber = topo.generate_amber(
    resp_input_file,
    frcmod_file=updated_frcmod,
    mol2_file=topology_mol2,
    output_prefix='SYS_new',
)
print(f"\u2713 Final AMBER topology: {new_amber['prmtop']}")

# Export
if final_format == "GROMACS":
    files = topo.generate_gromacs(new_amber['prmtop'], new_amber['inpcrd'])
    print(f"\u2713 GROMACS: {files['top']}")
elif final_format == "OpenMM":
    files = topo.generate_openmm(new_amber['prmtop'], new_amber['inpcrd'])
    print(f"\u2713 OpenMM: {files['xml']}")

print(f"\n\u2705 Done! Topology with Psi4 RESP charges + optimized dihedrals.")

---

## Download Results

In [ ]:
#@title **12. Package & Download Results**

import zipfile
from google.colab import files

zip_output = os.path.join(workDir, 'resp_topology_results.zip')

with zipfile.ZipFile(zip_output, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Include RESP charges file
    if os.path.exists(resp_result['output_file']):
        zf.write(resp_result['output_file'], 'resp_charges.dat')
    # Include topology files
    for key, path in amber_files.items():
        if path and os.path.exists(str(path)):
            zf.write(path, f'topology/{os.path.basename(path)}')
    # Include optimized structure
    if os.path.exists(opt_mol_file):
        zf.write(opt_mol_file, 'optimized_molecule.mol')
    if os.path.exists(opt_pdb_file):
        zf.write(opt_pdb_file, 'optimized_molecule.pdb')

print(f"\u2713 Results packaged: {zip_output}")
print(f"  Size: {os.path.getsize(zip_output)} bytes")

# Download in Colab
try:
    files.download(zip_output)
except:
    print(f"  (Download manually from: {zip_output})")